In [1]:
import re
import numpy as np
import pandas as pd
from typing import Dict, Optional, Tuple, Iterable

def compute_vertical_axis_map(
    df: pd.DataFrame,
    fs: float,
    method: str = "auto",      # "auto" | "quiet" | "lpf"
    fc: float = 1.0,           # LPF cutoff (Hz)
    win_s: float = 1.0,        # quiet-window length (s)
    omega_thr: float = 4.0,    # deg/s RMS threshold (quiet)
    a_thr: float = 0.8,        # m/s^2 RMS tolerance around g
    g: float = 9.81
) -> Dict[str, str]:
    """
    Returns a dict mapping sensor base name -> vertical axis letter ('x'|'y'|'z').

    Expects columns named "[position]_[optional side]_(acc|gyro)_[x|y|z]",
    e.g. "lowerback_acc_z", "foot_l_gyro_y".
    """
    sensors = _group_columns_by_sensor(df.columns)
    out: Dict[str, str] = {}

    for sensor, modes in sensors.items():
        acc_cols = modes.get("acc", {})
        gyro_cols = modes.get("gyro", {})

        # need full accel triplet to decide
        if set(acc_cols.keys()) != {"x","y","z"}:
            continue

        ax = df[acc_cols["x"]].to_numpy()
        ay = df[acc_cols["y"]].to_numpy()
        az = df[acc_cols["z"]].to_numpy()

        # choose method
        chosen = method
        if method == "auto":
            chosen = "quiet" if set(gyro_cols.keys()) == {"x","y","z"} else "lpf"

        if chosen == "quiet" and set(gyro_cols.keys()) == {"x","y","z"}:
            gx = df[gyro_cols["x"]].to_numpy()
            gy = df[gyro_cols["y"]].to_numpy()
            gz = df[gyro_cols["z"]].to_numpy()
            pick = _pick_axis_quiet(ax, ay, az, gx, gy, gz, fs, win_s, omega_thr, a_thr, g)
            if pick is None:
                k_axis = _pick_axis_lpf(ax, ay, az, fs, fc)
            else:
                k_axis = pick
        else:
            k_axis = _pick_axis_lpf(ax, ay, az, fs, fc)

        out[sensor] = ["x","y","z"][k_axis]

    return out

# ---------------- helpers ----------------

_sensor_pat = re.compile(r"^(?P<sensor>.+)_(?P<mode>acc|gyro)_(?P<axis>[xyz])$")

def _group_columns_by_sensor(columns) -> Dict[str, Dict[str, Dict[str, str]]]:
    sensors: Dict[str, Dict[str, Dict[str, str]]] = {}
    for c in columns:
        m = _sensor_pat.match(c)
        if not m:
            continue
        sensor = m.group("sensor")   # includes side if present, e.g. "foot_l"
        mode   = m.group("mode")     # acc|gyro
        axis   = m.group("axis")     # x|y|z
        sensors.setdefault(sensor, {}).setdefault(mode, {})[axis] = c
    return sensors

def _pick_axis_quiet(ax, ay, az, gx, gy, gz, fs, win_s, omega_thr, a_thr, g) -> Optional[int]:
    """
    Find a 'quiet' window (low rotation, accel magnitude near g),
    then choose the axis with largest |mean| in that window.
    Returns axis index (0=x,1=y,2=z) or None.
    """
    N = max(1, int(fs * win_s))
    A = np.column_stack([ax, ay, az])
    G = np.column_stack([gx, gy, gz])
    step = max(1, N // 4)

    for i in range(0, len(A) - N + 1, step):
        Aw = A[i:i+N]; Gw = G[i:i+N]
        w_rms = np.sqrt(np.mean(np.sum(Gw**2, axis=1)))  # deg/s RMS
        a_rms = np.sqrt(np.mean(np.sum(Aw**2, axis=1)))  # m/s^2 RMS
        if (w_rms < omega_thr) and (abs(a_rms - g) < a_thr):
            mu = Aw.mean(axis=0)
            return int(np.argmax(np.abs(mu)))
    return None

def _pick_axis_lpf(ax, ay, az, fs, fc) -> int:
    """
    Low-pass gravity estimate per axis and vote the most dominant axis.
    Returns axis index (0=x,1=y,2=z).
    """
    dt = 1.0 / fs
    alpha = (2*np.pi*fc*dt) / (1 + 2*np.pi*fc*dt)
    g_est = np.zeros(3)
    A = np.column_stack([ax, ay, az])
    votes = []

    for a in A:
        g_est = (1 - alpha) * g_est + alpha * a
        votes.append(int(np.argmax(np.abs(g_est))))

    if not votes:
        return 2  # default to 'z'
    ks = np.array(votes[len(votes)//4:])  # ignore early transient
    return int(np.bincount(ks).argmax())


def collapse_to_vertical_and_mag(
    df: pd.DataFrame,
    axis_map: Dict[str, str],
    *,
    kinds: Iterable[str] = ("acc", "gyro"),
    require_all_axes: bool = True,
    inplace: bool = False,
    lowercase: bool = True,
) -> pd.DataFrame:
    """
    For each sensor base in axis_map and each requested 'kind' (acc/gyro),
    produce exactly two columns:
        [base]_[kind]_v    : vertical component (rectified, abs)
        [base]_[kind]_mag  : magnitude of the two non-vertical axes

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe with columns like '[base]_[acc|gyro]_[x|y|z]'.
    axis_map : Dict[str, str]
        Mapping from sensor base -> vertical axis letter ('x'|'y'|'z'),
        e.g., {'lowerback':'z', 'foot_l':'y'}
        (You can get this from the `compute_vertical_axis_map` function you ran earlier.)
    kinds : Iterable[str], default ('acc','gyro')
        Which kinds to process.
    require_all_axes : bool, default True
        If True, only process when x,y,z exist for that base+kind.
        If False, proceed if the vertical axis and at least one non-vertical axis exist.
    inplace : bool, default False
        Modify input df in place.
    lowercase : bool, default True
        Match columns case-insensitively by lowering for lookups,
        but preserve original names when dropping.

    Returns
    -------
    pd.DataFrame
        DataFrame where, for each processed base+kind, the x/y/z columns are removed
        and replaced by two columns: '[base]_[kind]_v' and '[base]_[kind]_mag'.
    """
    work_df = df if inplace else df.copy()

    # Build a lookup from lowercased names to original names (to preserve original casing)
    name_lookup = { (c.lower() if lowercase else c): c for c in work_df.columns }

    def col_exists(base: str, kind: str, axis: str) -> Tuple[bool, str]:
        key = f"{base}_{kind}_{axis}"
        key_lc = key.lower() if lowercase else key
        if key_lc in name_lookup:
            return True, name_lookup[key_lc]
        return False, key  # return suggested name even if missing

    to_add = {}   # new_col_name -> Series
    to_drop = []  # original columns to remove

    for base, v_axis in axis_map.items():
        if v_axis not in ("x","y","z"):
            continue
        for kind in kinds:
            # Check which axes exist
            exist = {a: col_exists(base, kind, a) for a in ("x","y","z")}
            present_axes = [a for a, (ok, _) in exist.items() if ok]

            if require_all_axes:
                if set(present_axes) != {"x","y","z"}:
                    continue
            else:
                # Need vertical + at least one other axis to compute mag
                if not exist[v_axis][0]:
                    continue
                if len([a for a in ("x","y","z") if a != v_axis and exist[a][0]]) == 0:
                    continue

            # Vertical component (rectified)
            v_col = exist[v_axis][1]
            v_series = work_df[v_col].astype(float).abs()   # <-- rectified vertical

            # Magnitude of the other two axes
            other_axes = [a for a in ("x","y","z") if a != v_axis and exist[a][0]]
            block = work_df[[exist[a][1] for a in other_axes]].astype(float)
            mag_series = np.sqrt((block.pow(2)).sum(axis=1))

            out_v   = f"{base}_{kind}_v"
            out_mag = f"{base}_{kind}_mag"

            to_add[out_v] = v_series
            to_add[out_mag] = mag_series

            # Schedule original axes for drop
            to_drop.extend([exist[a][1] for a in present_axes])

    # Insert new columns
    for col, ser in to_add.items():
        work_df[col] = ser
    # Drop original axes
    if to_drop:
        work_df.drop(columns=sorted(set(to_drop)), inplace=True, errors="ignore")

    return work_df

In [3]:
import os
datasets = ['FOGSTAR', 'TDCS', 'IMU','Multimodal','Oday', 'rempark', 'Daphnet']
fs_dict = {'FOGSTAR':60,'TDCS':128, 'IMU': 128 ,'Multimodal': 500,'Oday':128, 'rempark': 200, 'Daphnet':64} 
os.chdir('Structured_datasets')
for data in datasets: 
    print('{} taken into analysis'.format(data))
    fs = float(fs_dict[data])
    raw = data +'.csv' 
    inp = data + "_filtered.csv"
    out = data + "_finalmag.csv" 
    df_raw = pd.read_csv(raw)
    axis_map = compute_vertical_axis_map(df_raw, fs) 
    del df_raw
    print('Computed vertical axes for {}'.format(data))
    df = pd.read_csv(inp)
    df_mag = collapse_to_vertical_and_mag(df, axis_map)
    df_mag.to_csv(out, index = False, mode = 'w') 
    print("{}'s components successfully collapsed".format(data))
os.chdir('..')

FOGSTAR taken into analysis
Computed vertical axes for FOGSTAR
FOGSTAR's components successfully collapsed
TDCS taken into analysis
Computed vertical axes for TDCS
TDCS's components successfully collapsed
IMU taken into analysis
Computed vertical axes for IMU
IMU's components successfully collapsed
Multimodal taken into analysis
Computed vertical axes for Multimodal
Multimodal's components successfully collapsed
Oday taken into analysis
Computed vertical axes for Oday
Oday's components successfully collapsed
rempark taken into analysis
Computed vertical axes for rempark
rempark's components successfully collapsed
Daphnet taken into analysis
Computed vertical axes for Daphnet
Daphnet's components successfully collapsed
